# HW2: ответы на тест за 180 с на одной T4

Модель: encoder-decoder Трансформер `MishaGustokashin`, обученный с нуля на `SlavaYus/tinysms_qa_12m`.
Итоговые веса `continue_9x9/avg.pt`: 9+9 слоёв, d = 512, 74.5M параметров, 9 ч обучения с нуля и 7.5 ч дообучения
на 2×T4, среднее 4 последних чекпойнтов. На каждый вопрос генерируется 12 программ, ответ выбирается голосованием.
На 2000 вопросах валидации жадная генерация даёт 0.7185, голосование 0.7570. Тест на T4 занимает около 140 с.

Settings: Accelerator GPU T4. Входы (Add Input): датасет `hw2-avg` (`tokenizer.json`, `val.jsonl`, `test.parquet`)
и output ноутбука обучения `continue` (`runs/continue_9x9/avg.pt`). Файлы ищутся по имени во всех подключённых входах.
`val.jsonl` нужен только для оценки, а `test.parquet` при его отсутствии скачивается с Hugging Face.

Как получается ответ. Вопрос проходит через BPE-токенизатор и энкодер (один раз на вопрос), затем декодер
с KV-cache пишет короткое тело программы (fp16, в батче вопросы похожей длины), и `denormalize` возвращает
заголовок `def solve() -> int:`, отступы и `return result`.

Голосование по исполнению. Из `VOTE_K` программ на вопрос одна жадная, остальные сэмплированы с температурой
`TEMPERATURE` и top-p `TOP_P`. Все кандидаты вопроса выполняются в одном процессе с таймаутом на программу,
пока GPU генерирует следующие батчи. Ответы группируются с тем же допуском, что у бота; в сабмит идёт программа
из самой большой группы, которой модель даёт наибольшую вероятность. Если ни одна программа не вернула число,
берётся жадная. Время голосования входит в замер. При `VOTE_K = 1` генерация обычная жадная.

Модели задаются строкой `MODELS` вида `'файл:K,файл:K'`: файл ищется по имени или хвосту пути во всех входах,
K это число кандидатов от этой модели. Если в `MODELS` несколько моделей, получается ансамбль: каждая модель даёт свою часть кандидатов, и голосуют
они вместе. В итоговой версии модель одна, ансамбль с другими весами на валидации оказался хуже.

## Код модели и генерации (встроен из `src/`)

In [1]:
import os
os.makedirs('src', exist_ok=True)

In [2]:
%%writefile src/model.py
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Config:
    vocab: int = 16000
    d: int = 512
    heads: int = 8
    n_enc: int = 6
    n_dec: int = 6
    ff_mult: int = 4
    max_src: int = 192
    max_tgt: int = 160
    ffn: str = 'gelu'

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.heads
        self.dh = cfg.d // cfg.heads
        self.q = nn.Linear(cfg.d, cfg.d, bias=False)
        self.kv = nn.Linear(cfg.d, 2 * cfg.d, bias=False)
        self.o = nn.Linear(cfg.d, cfg.d, bias=False)

    def heads_q(self, x):
        B, T, _ = x.shape
        return self.q(x).view(B, T, self.h, self.dh).transpose(1, 2)

    def heads_kv(self, src):
        B, S, _ = src.shape
        return self.kv(src).view(B, S, 2, self.h, self.dh).permute(2, 0, 3, 1, 4)

    def merge(self, y):
        B, _, T, _ = y.shape
        return self.o(y.transpose(1, 2).reshape(B, T, self.h * self.dh))

    def forward(self, x, mem=None, mask=None, causal=False):
        q = self.heads_q(x)
        k, v = self.heads_kv(x if mem is None else mem)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal)
        return self.merge(y)

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.swiglu = cfg.ffn == 'swiglu'
        if self.swiglu:
            hidden = round(2 / 3 * cfg.ff_mult * cfg.d / 64) * 64
            self.up = nn.Linear(cfg.d, 2 * hidden, bias=False)
        else:
            hidden = cfg.ff_mult * cfg.d
            self.up = nn.Linear(cfg.d, hidden, bias=False)
        self.down = nn.Linear(hidden, cfg.d, bias=False)

    def forward(self, x):
        if self.swiglu:
            gate, value = self.up(x).chunk(2, dim=-1)
            return self.down(F.silu(gate) * value)
        return self.down(F.gelu(self.up(x)))


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = nn.LayerNorm(cfg.d), Attention(cfg)
        self.n2, self.ffn = nn.LayerNorm(cfg.d), FFN(cfg)

    def forward(self, x, mask):
        x = x + self.attn(self.n1(x), mask=mask)
        x = x + self.ffn(self.n2(x))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.self_attn = nn.LayerNorm(cfg.d), Attention(cfg)
        self.n2, self.cross = nn.LayerNorm(cfg.d), Attention(cfg)
        self.n3, self.ffn = nn.LayerNorm(cfg.d), FFN(cfg)

    def forward(self, x, mem, mem_mask):
        x = x + self.self_attn(self.n1(x), causal=True)
        x = x + self.cross(self.n2(x), mem=mem, mask=mem_mask)
        x = x + self.ffn(self.n3(x))
        return x

    def step(self, x, t, cache_k, cache_v, cross_k, cross_v, mem_mask):
        """Один шаг генерации для нового токена x [B, 1, d] на позиции t.

        cache_k/cache_v [B, h, max_new, dh] хранят K и V уже сгенерированных токенов этого слоя;
        cross_k/cross_v [B, h, S, dh] посчитаны по выходу энкодера один раз на весь вопрос.
        """
        a = self.self_attn
        h = self.n1(x)
        k, v = a.heads_kv(h)
        cache_k[:, :, t:t + 1] = k
        cache_v[:, :, t:t + 1] = v
        y = F.scaled_dot_product_attention(a.heads_q(h), cache_k[:, :, :t + 1], cache_v[:, :, :t + 1])
        x = x + a.merge(y)

        c = self.cross
        y = F.scaled_dot_product_attention(c.heads_q(self.n2(x)), cross_k, cross_v, attn_mask=mem_mask)
        x = x + c.merge(y)
        return x + self.ffn(self.n3(x))


PAD, BOS, EOS = 0, 1, 2


def sample(logits, top_p=1.0, generator=None):
    """Один токен на строку из softmax(logits); top-p оставляет минимальный набор токенов с суммой вероятностей ≥ p."""
    probs = torch.softmax(logits, -1)
    if top_p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        drop = sp.cumsum(-1) - sp > top_p
        sp = sp.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, si, sp)
    return torch.multinomial(probs, 1, generator=generator)[:, 0]

class MishaGustokashin(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab, cfg.d)
        self.pos_src = nn.Embedding(cfg.max_src, cfg.d)
        self.pos_tgt = nn.Embedding(cfg.max_tgt, cfg.d)
        self.enc = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_enc)])
        self.dec = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_dec)])
        self.enc_norm = nn.LayerNorm(cfg.d)
        self.dec_norm = nn.LayerNorm(cfg.d)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, src):
        mask = (src != PAD)[:, None, None, :]
        pos = torch.arange(src.shape[1], device=src.device)
        x = self.emb(src) + self.pos_src(pos)

        for layer in self.enc:
            x = layer(x, mask)

        return self.enc_norm(x), mask

    def forward(self, src, tgt_in):
        mem, mask = self.encode(src)
        pos = torch.arange(tgt_in.shape[1], device=tgt_in.device)
        y = self.emb(tgt_in) + self.pos_tgt(pos)

        for layer in self.dec:
            y = layer(y, mem, mask)

        return self.logits(y)

    def logits(self, y):
        return F.linear(self.dec_norm(y), self.emb.weight)

    @torch.no_grad()
    def generate(self, src, max_new=96, temperature=0.0, top_p=1.0, return_logprob=False, generator=None):
        """Генерация с KV-cache: src [B, S] -> id кода [B, <= max_new], после <eos> идёт PAD.

        При temperature = 0 выбор жадный (argmax), при temperature > 0 токены сэмплируются из softmax(logits / T),
        по желанию с top-p. С return_logprob=True возвращается ещё сумма log p выбранных токенов по каждой строке,
        то есть уверенность модели в программе.
        """
        B = src.shape[0]
        mem, mask = self.encode(src)
        cross = [layer.cross.heads_kv(mem) for layer in self.dec]
        k0 = cross[0][0]
        shape = (B, self.cfg.heads, max_new, self.cfg.d // self.cfg.heads)
        caches = [(k0.new_zeros(shape), k0.new_zeros(shape)) for _ in self.dec]

        tok = torch.full((B,), BOS, dtype=torch.long, device=src.device)
        done = torch.zeros(B, dtype=torch.bool, device=src.device)
        out = []
        logprob = torch.zeros(B, device=src.device)
        for t in range(max_new):
            x = (self.emb(tok) + self.pos_tgt.weight[t])[:, None]
            for layer, (ck, cv), (xk, xv) in zip(self.dec, caches, cross):
                x = layer.step(x, t, ck, cv, xk, xv, mask)
            logits = self.logits(x)[:, -1].float()
            if temperature > 0:
                nxt = sample(logits / temperature, top_p, generator)
            else:
                nxt = logits.argmax(-1)
            if return_logprob:
                lp = torch.log_softmax(logits, -1).gather(1, nxt[:, None])[:, 0]
                logprob += lp.masked_fill(done, 0.0)
            nxt = nxt.masked_fill(done, PAD)
            out.append(nxt)
            done |= nxt == EOS
            tok = nxt
            if bool(done.all()):
                break
        ids = torch.stack(out, 1)
        return (ids, logprob) if return_logprob else ids

    @torch.no_grad()
    def generate_naive(self, src, max_new=96):
        """Эталон без кэша: на каждом шаге полный forward по всему префиксу. Только для проверки."""
        ys = torch.full((src.shape[0], 1), BOS, dtype=torch.long, device=src.device)
        for _ in range(max_new):
            nxt = self(src, ys)[:, -1].argmax(-1, keepdim=True)
            ys = torch.cat([ys, nxt], 1)
        return ys[:, 1:]

Writing src/model.py


In [3]:
%%writefile src/normalize.py
import ast
import io
import textwrap
import tokenize

HEADER = 'def solve() -> int:\n'
FOOTER = 'return result'


def _strip_comments(src):
    tokens = [t for t in tokenize.generate_tokens(io.StringIO(src).readline)
              if t.type != tokenize.COMMENT]
    return tokenize.untokenize(tokens)


def _non_empty_lines(src):
    return [line.rstrip() for line in src.split('\n') if line.strip()]


def normalize(code):
    """Короткое тело программы без заголовка, комментариев, print и return result.

    None, если программа не по шаблону."""
    if not code.startswith(HEADER):
        return None
    lines = _non_empty_lines(code[len(HEADER):])
    if not lines or lines[-1].strip() != FOOTER:
        return None
    body = textwrap.dedent('\n'.join(lines[:-1]))

    if '#' in body:
        try:
            body = '\n'.join(_non_empty_lines(_strip_comments(body + '\n')))
        except (tokenize.TokenError, SyntaxError):
            return None

    body = '\n'.join(line for line in body.split('\n') if not line.startswith('print('))

    try:
        ast.parse(body)
    except SyntaxError:
        return None
    return body


def denormalize(body):
    """Обратно к полной программе: заголовок def solve(), отступы и return result."""
    body = '\n'.join(_non_empty_lines(body))
    return HEADER + textwrap.indent(body, '    ') + '\n    ' + FOOTER + '\n'

Writing src/normalize.py


In [4]:
%%writefile src/execute.py
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
import numpy as np

RUNNER = r"""
import io, sys
code = sys.stdin.read()
real_stdout, sys.stdout = sys.stdout, io.StringIO()
try:
    ns = {}
    exec(code, ns)
    result = repr(float(ns['solve']()))
except Exception:
    result = ''
sys.stdout = real_stdout
print(result)
"""


def run_inline(code):
    """Выполняет программу в текущем процессе: число или None. Только для отладки."""
    try:
        ns = {}
        exec(code, ns)
        result = ns['solve']()
        return float(result)
    except Exception:
        return None


def run(code, timeout=2.0):
    """Выполняет программу в новом интерпретаторе; зависла дольше timeout -> None."""
    try:
        p = subprocess.run([sys.executable, '-I', '-c', RUNNER], input=code,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None
    try:
        return float(p.stdout.strip())
    except ValueError:
        return None


def is_correct(pred, gold):
    """Сравнение с тем же допуском, что у бота: rtol = atol = 1e-6. Нет ответа значит неверно."""
    if pred is None or gold is None:
        return False
    return abs(pred - gold) <= 1e-6 + 1e-6 * abs(gold)


def run_many(codes, workers=8):
    """Выполняет программы в workers потоках, каждую в своём процессе."""
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(run, codes))


def pass1(codes, golds):
    """Доля программ, чей ответ совпал с эталоном."""
    assert len(codes) == len(golds), (len(codes), len(golds))
    results = run_many(codes)
    return np.mean([is_correct(pred, gold) for pred, gold in zip(results, golds)])

Writing src/execute.py


In [5]:
%%writefile src/generate.py
"""Генерация программ по вопросам и файл сабмита.

    python src/generate.py --ckpt runs/base/best.pt --split val  --out val_pred.jsonl
    python src/generate.py --ckpt runs/base/best.pt --split test --out submission.jsonl
"""
import argparse
import dataclasses
import json
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src.model import EOS, PAD, Config, MishaGustokashin
from src.normalize import denormalize


def pick_device(name='auto'):
    if name != 'auto':
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def save_checkpoint(path, model, **extra):
    torch.save({'cfg': dataclasses.asdict(model.cfg), 'model': model.state_dict(), **extra}, path)


def load_model(path, device):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    model = MishaGustokashin(Config(**ck['cfg']))
    model.load_state_dict(ck['model'])
    return model.to(device).eval()


@torch.no_grad()
def generate_codes(model, tok, questions, device, batch_size=256, max_new=96):
    """Список вопросов -> список программ (в том же порядке)."""
    was_training = model.training
    model.eval()
    ids = [e.ids[:model.cfg.max_src] for e in tok.encode_batch(questions)]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    codes = [None] * len(ids)
    amp = torch.autocast('cuda', dtype=torch.float16) if device.type == 'cuda' else torch.autocast('cpu', enabled=False)

    for s in range(0, len(order), batch_size):
        batch = order[s:s + batch_size]
        L = max(1, max(len(ids[i]) for i in batch))
        src = torch.full((len(batch), L), PAD, dtype=torch.long)
        for r, i in enumerate(batch):
            src[r, :len(ids[i])] = torch.tensor(ids[i])
        with amp:
            out = model.generate(src.to(device), max_new=max_new).tolist()
        for r, i in enumerate(batch):
            row = out[r]
            row = row[:row.index(EOS)] if EOS in row else row
            codes[i] = denormalize(tok.decode(row))

    model.train(was_training)
    return codes


def main():
    import pandas as pd
    from tokenizers import Tokenizer
    from src.execute import pass1

    p = argparse.ArgumentParser()
    p.add_argument('--ckpt', required=True)
    p.add_argument('--data', type=Path, default=Path('data'))
    p.add_argument('--split', choices=['test', 'val'], default='test')
    p.add_argument('--out', required=True)
    p.add_argument('--limit', type=int, default=0, help='только первые N вопросов (для быстрых проверок)')
    p.add_argument('--batch_size', type=int, default=512)
    p.add_argument('--max_new', type=int, default=96)
    p.add_argument('--device', default='auto')
    a = p.parse_args()

    device = pick_device(a.device)
    tok = Tokenizer.from_file(str(a.data / 'tokenizer.json'))
    if a.split == 'test':
        questions = pd.read_parquet(a.data / 'raw' / 'test.parquet').question.tolist()
    else:
        val = [json.loads(line) for line in open(a.data / 'val.jsonl')]
        questions = [v['question'] for v in val]
    if a.limit:
        questions = questions[:a.limit]

    t0 = time.time()
    model = load_model(a.ckpt, device)
    codes = generate_codes(model, tok, questions, device, a.batch_size, a.max_new)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    print(f'{len(codes)} programs in {time.time() - t0:.1f} s on {device}')

    with open(a.out, 'w') as f:
        for c in codes:
            f.write(json.dumps({'code': c}) + '\n')
    if a.split == 'val':
        print(f'val pass@1 = {pass1(codes, [v["gold"] for v in val[:len(codes)]]):.4f}')


if __name__ == '__main__':
    main()

Writing src/generate.py


In [6]:
%%writefile src/vote.py
"""Голосование по результату исполнения (self-consistency).

На каждый вопрос генерируется K программ (первая жадная, остальные сэмплированы с температурой),
все выполняются, ответы группируются с допуском бота, побеждает самая большая группа; в сабмит идёт
программа из неё, которой модель даёт наибольшую вероятность. Если ни одна программа не вернула число,
берётся жадная. Голосование работает только на инференсе, обучение оно не затрагивает.

    python src/vote.py --ckpt runs/base_d512_6x6/best.pt --split val --k 16 --temperature 0.7 --limit 500
"""
import argparse
import json
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src.execute import is_correct
from src.model import EOS, PAD
from src.normalize import denormalize

RUNNER_MANY = r"""
import io, json, math, signal, sys
codes = json.loads(sys.stdin.read())
timeout = float(sys.argv[1])
real_stdout = sys.stdout

class Timeout(BaseException):
    pass

def on_alarm(*_):
    raise Timeout()

signal.signal(signal.SIGALRM, on_alarm)
out = []
for code in codes:
    sys.stdout = io.StringIO()
    try:
        signal.setitimer(signal.ITIMER_REAL, timeout)
        ns = {}
        exec(code, ns)
        r = float(ns['solve']())
        if math.isnan(r) or math.isinf(r):
            r = None
    except BaseException:
        r = None
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
    out.append(r)
sys.stdout = real_stdout
print(json.dumps(out))
"""


def run_group(codes, timeout=1.0):
    """Список программ -> список ответов (float | None), всё в одном процессе."""
    try:
        p = subprocess.run([sys.executable, '-I', '-c', RUNNER_MANY, str(timeout)], input=json.dumps(codes),
                           capture_output=True, text=True, timeout=timeout * len(codes) + 5)
        res = json.loads(p.stdout.strip().splitlines()[-1])
        return res if len(res) == len(codes) else [None] * len(codes)
    except Exception:
        return [None] * len(codes)


def run_groups(groups, timeout=1.0, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(lambda g: run_group(g, timeout), groups))


@torch.no_grad()
def generate_scored(model, tok, questions, device, batch_size=512, max_new=128,
                    temperature=0.0, top_p=1.0, seed=0, on_batch=None):
    """Как generate_codes, но с параметрами сэмплирования; возвращает [(code, logprob)] в порядке вопросов.

    on_batch(список (индекс вопроса, (code, logprob))) вызывается после каждого батча, чтобы CPU исполнял
    готовые программы, пока GPU генерирует следующий батч.
    """
    model.eval()
    gen = torch.Generator(device=device).manual_seed(seed) if temperature > 0 else None
    ids = [e.ids[:model.cfg.max_src] for e in tok.encode_batch(questions)]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    res = [None] * len(ids)
    amp = torch.autocast('cuda', dtype=torch.float16) if device.type == 'cuda' else torch.autocast('cpu', enabled=False)
    for s in range(0, len(order), batch_size):
        batch = order[s:s + batch_size]
        L = max(1, max(len(ids[i]) for i in batch))
        src = torch.full((len(batch), L), PAD, dtype=torch.long)
        for r, i in enumerate(batch):
            src[r, :len(ids[i])] = torch.tensor(ids[i])
        with amp:
            out, lp = model.generate(src.to(device), max_new=max_new, temperature=temperature, top_p=top_p,
                                     return_logprob=True, generator=gen)
        out, lp = out.tolist(), lp.tolist()
        for r, i in enumerate(batch):
            row = out[r]
            row = row[:row.index(EOS)] if EOS in row else row
            res[i] = (denormalize(tok.decode(row)), lp[r])
        if on_batch:
            on_batch([(i, res[i]) for i in batch])
    return res


def candidates(model, tok, questions, device, k, temperature=0.7, top_p=0.95, batch_size=512, max_new=128, seed=0):
    """K кандидатов на вопрос: [жадный] + (K-1) сэмплов. Возвращает список списков (code, logprob)."""
    greedy = generate_scored(model, tok, questions, device, batch_size, max_new)
    cands = [[g] for g in greedy]
    if k > 1:
        rep = [q for q in questions for _ in range(k - 1)]
        sampled = generate_scored(model, tok, rep, device, batch_size, max_new, temperature, top_p, seed)
        for i in range(len(questions)):
            cands[i] += sampled[i * (k - 1):(i + 1) * (k - 1)]
    return cands


def candidates_and_values(model, tok, questions, device, k, temperature=0.7, top_p=0.95, batch_size=512,
                          max_new=128, seed=0, timeout=1.0, workers=8):
    """То же, что candidates + run_groups, но исполнение идёт параллельно с генерацией: как только у вопроса
    готовы все K кандидатов, они уходят на исполнение в фоновый поток, а GPU генерирует дальше."""
    n = len(questions)
    pool = ThreadPoolExecutor(max_workers=workers)
    futures = [None] * n
    cands = [[g] for g in generate_scored(model, tok, questions, device, batch_size, max_new)]
    if k == 1:
        futures = [pool.submit(run_group, [cs[0][0]], timeout) for cs in cands]
    else:
        need = k - 1
        samples = [[None] * need for _ in range(n)]
        got = [0] * n

        def on_batch(items):
            for j, item in items:
                i, s = divmod(j, need)
                samples[i][s] = item
                got[i] += 1
                if got[i] == need:
                    cands[i] += samples[i]
                    futures[i] = pool.submit(run_group, [c for c, _ in cands[i]], timeout)

        rep = [q for q in questions for _ in range(need)]
        generate_scored(model, tok, rep, device, batch_size, max_new, temperature, top_p, seed, on_batch=on_batch)
    values = [f.result() for f in futures]
    pool.shutdown()
    return cands, values


def ensemble_candidates_and_values(models, ks, tok, questions, device, **kw):
    """Кандидаты от нескольких моделей: i-я модель даёт ks[i] штук (жадный и ks[i] - 1 сэмплов), списки
    склеиваются и голосуют вместе. Если ни одна программа не вернула число, в ответ идёт жадная программа
    первой модели."""
    cands, values = [[] for _ in questions], [[] for _ in questions]
    for model, k in zip(models, ks):
        c, v = candidates_and_values(model, tok, questions, device, k, **kw)
        for i in range(len(questions)):
            cands[i] += c[i]
            values[i] += v[i]
    return cands, values


def vote(cands, values):
    """cands: [(code, logprob)], values: [float | None] -> (выбранная программа, её ответ | None)."""
    groups = []
    for (code, lp), v in zip(cands, values):
        if v is None:
            continue
        for g in groups:
            if is_correct(v, g['value']):
                g['n'] += 1
                g['lp_sum'] += lp
                if lp > g['best_lp']:
                    g['best_lp'], g['code'] = lp, code
                break
        else:
            groups.append({'value': v, 'n': 1, 'lp_sum': lp, 'best_lp': lp, 'code': code})
    if not groups:
        return cands[0][0], None
    best = max(groups, key=lambda g: (g['n'], g['lp_sum']))
    return best['code'], best['value']


def main():
    from tokenizers import Tokenizer
    from src.generate import load_model, pick_device

    p = argparse.ArgumentParser()
    p.add_argument('--ckpt', required=True)
    p.add_argument('--data', type=Path, default=Path('data'))
    p.add_argument('--split', choices=['val'], default='val')
    p.add_argument('--limit', type=int, default=0)
    p.add_argument('--k', type=int, default=16)
    p.add_argument('--temperature', type=float, default=0.7)
    p.add_argument('--top_p', type=float, default=0.95)
    p.add_argument('--batch_size', type=int, default=512)
    p.add_argument('--max_new', type=int, default=128)
    p.add_argument('--device', default='auto')
    p.add_argument('--cache', type=Path, default=None, help='сохранить кандидатов и ответы (jsonl) для анализа')
    a = p.parse_args()

    device = pick_device(a.device)
    tok = Tokenizer.from_file(str(a.data / 'tokenizer.json'))
    val = [json.loads(line) for line in open(a.data / 'val.jsonl')]
    if a.limit:
        val = val[:a.limit]
    questions, golds = [v['question'] for v in val], [v['gold'] for v in val]
    model = load_model(a.ckpt, device)

    t0 = time.time()
    cands = candidates(model, tok, questions, device, a.k, a.temperature, a.top_p, a.batch_size, a.max_new)
    t_gen = time.time() - t0
    t0 = time.time()
    values = run_groups([[c for c, _ in cs] for cs in cands])
    t_exec = time.time() - t0
    print(f'{len(questions)} вопросов × K={a.k}, T={a.temperature}, top_p={a.top_p}: '
          f'генерация {t_gen:.0f} с, исполнение {t_exec:.0f} с ({device})')

    ks = [k for k in [1, 2, 4, 8, 16, 32] if k <= a.k]
    for k in ks:
        voted = [vote(cs[:k], vs[:k])[1] for cs, vs in zip(cands, values)]
        p_vote = sum(is_correct(v, g) for v, g in zip(voted, golds)) / len(golds)
        p_any = sum(any(is_correct(v, g) for v in vs[:k]) for vs, g in zip(values, golds)) / len(golds)
        print(f'K={k:2d}: голосование {p_vote:.4f} | хотя бы один верный (верхняя граница) {p_any:.4f}')

    if a.cache:
        with open(a.cache, 'w') as f:
            for q, g, cs, vs in zip(questions, golds, cands, values):
                f.write(json.dumps({'question': q, 'gold': g, 'codes': [c for c, _ in cs],
                                    'logprobs': [lp for _, lp in cs], 'values': vs}) + '\n')


if __name__ == '__main__':
    main()

Writing src/vote.py


## Входные данные

In [7]:
import glob, json, os, sys, time, urllib.request
import pandas as pd, torch
from tokenizers import Tokenizer
sys.path.insert(0, '.')
from src.generate import load_model, generate_codes
from src.vote import ensemble_candidates_and_values, vote
from src.execute import is_correct, pass1

INPUT = os.environ.get('HW2_INPUT', '/kaggle/input')
MODELS = [(name, int(k)) for name, k in (x.split(':') for x in os.environ.get('HW2_MODELS', 'continue_9x9/avg.pt:12').split(','))]
VOTE_K = sum(k for _, k in MODELS)


def find(name):
    """Файл с таким именем в любом из подключённых входов (или None)."""
    hits = sorted(glob.glob(f'{INPUT}/**/{name}', recursive=True))
    return hits[0] if hits else None


CKPTS = [find(name) for name, _ in MODELS]
TOKENIZER, VAL, TEST = find('tokenizer.json'), find('val.jsonl'), find('test.parquet')
assert all(CKPTS) and TOKENIZER, f'нужны {[n for n, _ in MODELS]} и tokenizer.json во входных данных: {CKPTS}, {TOKENIZER}'
if TEST is None:
    TEST = 'test.parquet'
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/main/data/test/test.parquet', TEST)
questions = pd.read_parquet(TEST).question.tolist()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for k, v in [*[(f'ckpt ×{k}', c) for (_, k), c in zip(MODELS, CKPTS)], ('tokenizer', TOKENIZER), ('val', VAL), ('test', TEST)]:
    print(f'{k:9s} {v}')
print(len(questions), 'questions |', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu')

ckpt ×12  /kaggle/input/notebooks/lisich257/continue/runs/continue_9x9/avg.pt
tokenizer /kaggle/input/datasets/lisich257/hw2-avg/tokenizer.json
val       /kaggle/input/datasets/lisich257/hw2-avg/val.jsonl
test      /kaggle/input/datasets/lisich257/hw2-avg/test.parquet
1919 questions | Tesla T4


## Генерация теста и замер времени
В замер входит всё: загрузка токенизатора и модели, генерация всех кандидатов, их исполнение,
голосование и запись файла. Температура и top-p подобраны на валидации. Батч большой (2048), потому что
шаг декодера на T4 упирается в накладные расходы, а не в вычисления.

In [8]:
TEMPERATURE, TOP_P = 0.7, 0.95
MAX_NEW, BATCH = 128, 2048


def predict(qs):
    """Вопросы -> (программы, (кандидаты, их ответы) или None, тайминги)."""
    T, t = {}, time.time()
    if VOTE_K == 1:
        codes = generate_codes(models[0], tok, qs, device, batch_size=BATCH, max_new=MAX_NEW)
        T['генерация'] = time.time() - t
        return codes, None, T
    cands, values = ensemble_candidates_and_values(models, [k for _, k in MODELS], tok, qs, device,
                                                   temperature=TEMPERATURE, top_p=TOP_P,
                                                   batch_size=BATCH, max_new=MAX_NEW)
    T['генерация + исполнение'], t = time.time() - t, time.time()
    codes = [vote(cs, vs)[0] for cs, vs in zip(cands, values)]
    T['голосование'] = time.time() - t
    return codes, (cands, values), T


t0 = time.time()
tok = Tokenizer.from_file(TOKENIZER)
models = [load_model(c, device) for c in CKPTS]
t_load = time.time() - t0
codes, _, T = predict(questions)
with open('submission.jsonl', 'w') as f:
    for c in codes:
        f.write(json.dumps({'code': c}) + '\n')
elapsed = time.time() - t0
parts = ' · '.join(f'{k} {v:.1f} с' for k, v in {'загрузка': t_load, **T}.items())
print(f'{len(codes)} programs in {elapsed:.1f} s  (лимит 180 с; MODELS={MODELS}: {parts})')
print(codes[0])

1919 programs in 139.6 s  (лимит 180 с; MODELS=[('continue_9x9/avg.pt', 12)]: загрузка 4.9 с · генерация + исполнение 134.7 с · голосование 0.0 с)
def solve() -> int:
    price_before_tax = 54 / 1.1
    result = price_before_tax
    return result



## Проверка сабмита (не входит в замер)

In [9]:
import ast
from src.execute import run_many

def parses(src):
    try:
        ast.parse(src)
        return True
    except SyntaxError:
        return False

rows = [json.loads(line) for line in open('submission.jsonl')]
assert len(rows) == len(questions) == 1919, len(rows)
ok_syntax = sum(parses(r['code']) for r in rows)
values = run_many([r['code'] for r in rows])
numeric = sum(v is not None for v in values)
print(f'строк {len(rows)}, синтаксис ок {ok_syntax}, вернули число {numeric} ({numeric / len(rows):.1%})')

строк 1919, синтаксис ок 1919, вернули число 1917 (99.9%)


## Качество на валидации (не входит в замер)

In [10]:
if VAL:
    val = [json.loads(line) for line in open(VAL)]
    golds = [v['gold'] for v in val]
    t = time.time()
    val_codes, extra, T = predict([v['question'] for v in val])
    line = f'val: {len(val)} вопросов за {time.time() - t:.1f} с, pass@1 = {pass1(val_codes, golds):.4f}'
    if extra:
        cands, values = extra
        greedy = sum(is_correct(vs[0], g) for vs, g in zip(values, golds)) / len(golds)
        line += f' (голосование, K={VOTE_K}) | жадно (первая модель) {greedy:.4f}'
    print(line)
else:
    print('val.jsonl нет во входных данных, пропускаю')

val: 2000 вопросов за 159.8 с, pass@1 = 0.7570 (голосование, K=12) | жадно (первая модель) 0.7185


## Сравнение вариантов на валидации (не входит в сабмит)
Строки `COMPARE` записываются так же, как `MODELS`: `'файл:K,файл:K'`. Если файлов с одним именем несколько,
вместо имени пишется хвост пути (`final2/avg.pt`). Время на 2000 вопросах валидации примерно равно времени
на 1919 вопросах теста. Модели с 12 слоями декодера не помещаются в память T4 с батчем 2048, поэтому при нехватке
памяти батч уменьшается вдвое. Этим сравнением выбиралась итоговая конфигурация: K = 8 дал 0.7470, ансамбль
`final2/avg.pt` ×4 + `hw2-avg/avg.pt` ×4 дал 0.7345, K = 12 дал 0.7570.

In [11]:
COMPARE = []
if COMPARE and VAL:
    val = [json.loads(line) for line in open(VAL)]
    vq, golds = [v['question'] for v in val], [v['gold'] for v in val]
    for spec in COMPARE:
        ms = [(name, int(k)) for name, k in (x.split(':') for x in spec.split(','))]
        paths = [find(name) for name, _ in ms]
        if not all(paths):
            print(f'{spec}: не найден файл {paths}')
            continue
        mods = [load_model(p_, device) for p_ in paths]
        for bs in (BATCH, BATCH // 2, BATCH // 4):
            try:
                t = time.time()
                c, v = ensemble_candidates_and_values(mods, [k for _, k in ms], tok, vq, device, temperature=TEMPERATURE,
                                                      top_p=TOP_P, batch_size=bs, max_new=MAX_NEW)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print(f'{spec}: батч {bs} не влез в память, пробую {bs // 2}')
        voted = [vote(cs, vs)[1] for cs, vs in zip(c, v)]
        dt = time.time() - t
        acc = sum(is_correct(x, g) for x, g in zip(voted, golds)) / len(golds)
        greedy = sum(is_correct(vs[0], g) for vs, g in zip(v, golds)) / len(golds)
        print(f'{spec:55s} голосование {acc:.4f} | жадно (первая) {greedy:.4f} | {dt:.0f} с на {len(vq)} вопросов (батч {bs})')
        del mods, c, v
        torch.cuda.empty_cache() if device.type == 'cuda' else None